# Google Analytics Performance Report

This notebook demonstrates the enhanced Google Analytics reporting capabilities in siege_utilities.

## Features Demonstrated

1. **KPI Dashboard** - Metric cards with period-over-period comparison
2. **Traffic Trends** - Time series visualization
3. **Traffic Sources** - Pie charts and detailed tables
4. **Geographic Analysis** - Choropleth maps, heatmaps, Census integration
5. **Page Performance** - Top pages with engagement metrics
6. **Automated Insights** - AI-generated analysis
7. **PDF Generation** - Professional multi-page reports

In [ ]:
# Imports
import sys
from datetime import datetime, timedelta
from pathlib import Path

# Add parent directory for imports if running from notebooks/
sys.path.insert(0, str(Path.cwd().parent))

import siege_utilities as su
su.configure_shared_logging(level="INFO")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 1. Generate Sample GA Data

In production, you would use `GoogleAnalyticsConnector` to fetch real data.
For demonstration, we'll generate realistic sample data.

In [ ]:
from siege_utilities.reporting.examples.google_analytics_report_example import (
    generate_sample_ga_data,
    create_kpi_dashboard,
    create_traffic_trend_chart,
    create_traffic_sources_chart,
    generate_insights,
    generate_recommendations,
    generate_ga_report_pdf
)

# Generate 30 days of sample data
end_date = datetime.now()
start_date = end_date - timedelta(days=30)

ga_data = generate_sample_ga_data(start_date, end_date)

su.log_info(f"Date Range: {ga_data['date_range']['start']} to {ga_data['date_range']['end']}")
su.log_info("Key Metrics:")
su.log_info(f"  Total Users: {ga_data['totals']['users']:,}")
su.log_info(f"  Total Sessions: {ga_data['totals']['sessions']:,}")
su.log_info(f"  Avg Bounce Rate: {ga_data['totals']['avg_bounce_rate']:.1f}%")
su.log_info(f"  Avg Session Duration: {ga_data['totals']['avg_session_duration']:.0f}s")
su.log_info("Period-over-Period Changes:")
su.log_info(f"  Users: {ga_data['changes']['users']:+.1f}%")
su.log_info(f"  Sessions: {ga_data['changes']['sessions']:+.1f}%")

## 2. Daily Traffic Visualization

In [ ]:
# Create daily data DataFrame
daily_df = pd.DataFrame({
    'date': pd.to_datetime(ga_data['daily_data']['dates']),
    'users': ga_data['daily_data']['users'],
    'sessions': ga_data['daily_data']['sessions'],
    'pageviews': ga_data['daily_data']['pageviews'],
    'bounce_rate': ga_data['daily_data']['bounce_rate']
})

# Plot traffic trends
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Users and Sessions
ax1 = axes[0, 0]
ax1.plot(daily_df['date'], daily_df['users'], label='Users', color='#3366cc', linewidth=2)
ax1.plot(daily_df['date'], daily_df['sessions'], label='Sessions', color='#dc3912', linewidth=2, alpha=0.7)
ax1.set_title('Daily Users & Sessions', fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.tick_params(axis='x', rotation=45)

# Pageviews
ax2 = axes[0, 1]
ax2.fill_between(daily_df['date'], daily_df['pageviews'], alpha=0.5, color='#109618')
ax2.plot(daily_df['date'], daily_df['pageviews'], color='#109618', linewidth=2)
ax2.set_title('Daily Pageviews', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.tick_params(axis='x', rotation=45)

# Bounce Rate
ax3 = axes[1, 0]
ax3.bar(daily_df['date'], daily_df['bounce_rate'], color='#ff9900', alpha=0.7)
ax3.axhline(y=daily_df['bounce_rate'].mean(), color='red', linestyle='--', label='Average')
ax3.set_title('Daily Bounce Rate', fontsize=12, fontweight='bold')
ax3.set_ylabel('Bounce Rate (%)')
ax3.legend()
ax3.tick_params(axis='x', rotation=45)

# Weekly pattern
ax4 = axes[1, 1]
daily_df['weekday'] = daily_df['date'].dt.day_name()
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekly_avg = daily_df.groupby('weekday')['users'].mean().reindex(weekday_order)
ax4.bar(range(7), weekly_avg.values, color='#990099')
ax4.set_xticks(range(7))
ax4.set_xticklabels(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])
ax4.set_title('Average Users by Day of Week', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Traffic Sources Analysis

In [ ]:
# Traffic sources DataFrame
sources_df = pd.DataFrame(ga_data['traffic_sources'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
colors = ['#3366cc', '#dc3912', '#ff9900', '#109618', '#990099']
wedges, texts, autotexts = axes[0].pie(
    sources_df['sessions'],
    labels=sources_df['source'].str.title(),
    autopct='%1.1f%%',
    colors=colors,
    startangle=90
)
axes[0].set_title('Traffic Sources Distribution', fontsize=12, fontweight='bold')

# Bar chart with bounce rate
x = range(len(sources_df))
width = 0.35
bars1 = axes[1].bar([i - width/2 for i in x], sources_df['sessions']/1000, width, label='Sessions (K)', color='#3366cc')
ax2 = axes[1].twinx()
bars2 = ax2.bar([i + width/2 for i in x], sources_df['bounce_rate'], width, label='Bounce Rate (%)', color='#dc3912', alpha=0.7)

axes[1].set_xticks(x)
axes[1].set_xticklabels(sources_df['source'].str.title())
axes[1].set_ylabel('Sessions (thousands)', color='#3366cc')
ax2.set_ylabel('Bounce Rate (%)', color='#dc3912')
axes[1].set_title('Sessions vs Bounce Rate by Source', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

# Display table
su.log_info("Traffic Sources Summary:")
display_df = sources_df.copy()
display_df['sessions'] = display_df['sessions'].apply(lambda x: f"{x:,}")
display_df['users'] = display_df['users'].apply(lambda x: f"{x:,}")
display_df['bounce_rate'] = display_df['bounce_rate'].apply(lambda x: f"{x:.1f}%")
display_df['avg_duration'] = display_df['avg_duration'].apply(lambda x: f"{x:.1f}s")
su.log_info(display_df.to_string(index=False))

## 4. Top Pages Performance

In [ ]:
# Top pages DataFrame
pages_df = pd.DataFrame(ga_data['top_pages'])

fig, ax = plt.subplots(figsize=(12, 5))

# Horizontal bar chart
y_pos = range(len(pages_df))
bars = ax.barh(y_pos, pages_df['pageviews'], color='#3366cc')

ax.set_yticks(y_pos)
ax.set_yticklabels(pages_df['page'])
ax.set_xlabel('Pageviews')
ax.set_title('Top Pages by Pageviews', fontsize=12, fontweight='bold')

# Add value labels
for i, (bar, val) in enumerate(zip(bars, pages_df['pageviews'])):
    ax.text(val + max(pages_df['pageviews'])*0.01, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

# Display table
su.log_info("Top Pages Performance:")
display_pages = pages_df.copy()
display_pages['pageviews'] = display_pages['pageviews'].apply(lambda x: f"{x:,}")
display_pages['avg_time'] = display_pages['avg_time'].apply(lambda x: f"{x:.1f}s")
display_pages['bounce_rate'] = display_pages['bounce_rate'].apply(lambda x: f"{x:.1f}%")
display_pages['exit_rate'] = display_pages['exit_rate'].apply(lambda x: f"{x:.1f}%")
su.log_info(display_pages.to_string(index=False))

## 5. Device Analysis

In [ ]:
# Device breakdown
devices_df = pd.DataFrame(ga_data['devices'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pie chart
colors = ['#3366cc', '#dc3912', '#ff9900']
axes[0].pie(
    devices_df['sessions'],
    labels=devices_df['device'].str.title(),
    autopct='%1.1f%%',
    colors=colors,
    startangle=90
)
axes[0].set_title('Sessions by Device', fontsize=12, fontweight='bold')

# Bounce rate comparison
bars = axes[1].bar(devices_df['device'].str.title(), devices_df['bounce_rate'], color=colors)
axes[1].set_ylabel('Bounce Rate (%)')
axes[1].set_title('Bounce Rate by Device', fontsize=12, fontweight='bold')

# Add value labels
for bar, val in zip(bars, devices_df['bounce_rate']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{val:.1f}%', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

## 6. Geographic Distribution

In [ ]:
# Geographic data
geo_df = pd.DataFrame(ga_data['geo_data'])

fig, ax = plt.subplots(figsize=(10, 5))

# Bar chart of top cities
colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(geo_df)))
bars = ax.barh(range(len(geo_df)), geo_df['sessions'], color=colors)

ax.set_yticks(range(len(geo_df)))
ax.set_yticklabels([f"{row['city']}, {row['region']}" for _, row in geo_df.iterrows()])
ax.set_xlabel('Sessions')
ax.set_title('Top Cities by Sessions', fontsize=12, fontweight='bold')

# Add value labels
for bar, val in zip(bars, geo_df['sessions']):
    ax.text(val + max(geo_df['sessions'])*0.01, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

## 7. Automated Insights & Recommendations

In [ ]:
# Generate insights
insights = generate_insights(ga_data)
recommendations = generate_recommendations(ga_data)

su.log_info("Key Insights:")
su.log_info("=" * 50)
for i, insight in enumerate(insights, 1):
    su.log_info(f"{i}. {insight}")

su.log_info("Recommendations:")
su.log_info("=" * 50)
for i, rec in enumerate(recommendations, 1):
    su.log_info(f"{i}. {rec}")

## 8. Generate PDF Report

In [ ]:
# Generate comprehensive PDF report
output_path = "ga_performance_report.pdf"

success = generate_ga_report_pdf(
    ga_data=ga_data,
    output_path=output_path,
    client_name="Demo Company",
    report_title="Website Analytics Report"
)

if success:
    su.log_info(f"Report generated successfully: {output_path}")
    su.log_info("Report includes:")
    su.log_info("  - Title Page")
    su.log_info("  - Executive Summary")
    su.log_info("  - KPI Dashboard with metric cards")
    su.log_info("  - Traffic Trends chart")
    su.log_info("  - Traffic Sources analysis")
    su.log_info("  - Device breakdown")
    su.log_info("  - Top Pages performance table")
    su.log_info("  - Geographic distribution")
    su.log_info("  - Key Insights")
    su.log_info("  - Recommendations")
else:
    su.log_error("Report generation failed. Check logs for details.")

## 9. Using Real Google Analytics Data

To use real GA4 data, use the `GoogleAnalyticsConnector`:

```python
from siege_utilities.analytics import GoogleAnalyticsConnector

# Option 1: Service Account (recommended for automation)
ga = GoogleAnalyticsConnector(
    auth_method="service_account",
    service_account_data=service_account_json
)

# Option 2: From 1Password
from siege_utilities.analytics import create_ga_connector_from_1password
ga = create_ga_connector_from_1password("Google Analytics Service Account")

# Fetch GA4 data
df = ga.get_ga4_data(
    property_id="123456789",
    start_date="2025-01-01",
    end_date="2025-01-31",
    metrics=["sessions", "users", "bounceRate", "averageSessionDuration"],
    dimensions=["date", "city", "deviceCategory"]
)
```

## 10. Geographic Analysis with Census Data

For advanced geographic analysis, use the Census integration:

```python
from siege_utilities.reporting.examples.ga_geographic_analysis import (
    geocode_ga_cities,
    aggregate_by_state,
    create_state_choropleth,
    create_traffic_demographics_comparison
)

# Geocode GA city data
ga_df = geocode_ga_cities(ga_city_data)

# Aggregate to state level
state_df = aggregate_by_state(ga_df)

# Create choropleth map
choropleth_path = create_state_choropleth(state_df, 'sessions')

# Join with Census demographics
merged = create_traffic_demographics_comparison(state_df, census_year=2022)
```

## Summary

This notebook demonstrated:

1. **Sample Data Generation** - Realistic GA data patterns
2. **Traffic Visualization** - Line charts, bar charts, weekly patterns
3. **Source Analysis** - Pie charts, comparative bar charts
4. **Page Performance** - Top pages with engagement metrics
5. **Device Analysis** - Mobile vs desktop comparison
6. **Geographic Distribution** - City-level traffic
7. **Automated Insights** - AI-generated analysis
8. **PDF Generation** - Professional multi-page reports

For production use, connect to real GA4 data using `GoogleAnalyticsConnector`.